# ДЗ. Нейрон с функцией активации tanh

Старинская Светлана, 3ФИИТ

## Подготовка окружения

Используется датасет «Яблоки и груши» из ЛР3. В работе не нужны дополнительные библиотеки: достаточно NumPy и Matplotlib.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

DATA_PATH = Path('apples_pears.csv')
SEED = 42
TRAIN_SHARE = 0.8
LEARNING_RATE = 0.03
EPOCHS = 5000

np.random.seed(SEED)

## Задание. Нейрон с функцией tanh

В ЛР3 были рассмотрены пороговая функция и сигмоида. Здесь реализуется нейрон с гиперболическим тангенсом:

$$tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}, tanh'(z) = 1 - tanh^2(z).$$

Tanh возвращает значения от −1 до 1, поэтому для обучения метки 0 и 1 временно переводятся в −1 и 1.

In [ ]:
class TanhNeuron:
    def __init__(self, learning_rate=LEARNING_RATE):
        self.learning_rate = learning_rate
        self.w = None
        self.b = 0.0

    def forward_pass(self, X):
        with np.errstate(over='ignore', divide='ignore', invalid='ignore'):
            z = np.clip(X @ self.w + self.b, -50, 50)
        return np.tanh(z)

    @staticmethod
    def loss(y_pred, y):
        return np.mean((y_pred - y) ** 2)

    def backward_pass(self, X, y, y_pred):
        error = (y_pred - y) * (1 - y_pred ** 2)
        grad_w = 2 / len(X) * X.T @ error
        grad_b = 2 * error.mean()
        self.w -= self.learning_rate * grad_w
        self.b -= self.learning_rate * grad_b

    def fit(self, X, y, num_epochs=EPOCHS):
        self.w = np.zeros((X.shape[1], 1))
        self.b = 0.0
        loss_values = []

        for _ in range(num_epochs):
            y_pred = self.forward_pass(X)
            loss_values.append(self.loss(y_pred, y))
            self.backward_pass(X, y, y_pred)

        return loss_values

    def predict(self, X):
        return (self.forward_pass(X) >= 0).astype(int)

## Загрузка и визуализация данных

Признаки: желтизна и симметричность. Целевой класс находится в столбце `target`.

In [ ]:
data = np.genfromtxt(DATA_PATH, delimiter=',', skip_header=1)
X = data[:, :2]
y = data[:, 2].astype(int).reshape(-1, 1)

print(f'Количество объектов: {len(X)}')
print(f'Класс 0: {(y == 0).sum()} | класс 1: {(y == 1).sum()}')

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap='coolwarm', edgecolor='black', alpha=0.7)
plt.title('Яблоки и груши: исходные классы')
plt.xlabel('Желтизна')
plt.ylabel('Симметричность')
plt.grid(alpha=0.25)
plt.show()

## Обучение и тестирование

Делим данные на train и test в пропорции 80/20. Нормализация выполняется только по train-части, чтобы тестовые данные не влияли на обучение.

In [ ]:
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(X))
split_index = int(len(X) * TRAIN_SHARE)
train_idx, test_idx = indices[:split_index], indices[split_index:]

X_train_raw, X_test_raw = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

mean = X_train_raw.mean(axis=0)
std = X_train_raw.std(axis=0)
X_train = (X_train_raw - mean) / std
X_test = (X_test_raw - mean) / std

y_train_tanh = 2 * y_train - 1
neuron = TanhNeuron()
loss_values = neuron.fit(X_train, y_train_tanh)

train_accuracy = (neuron.predict(X_train) == y_train).mean()
test_accuracy = (neuron.predict(X_test) == y_test).mean()

print(f'Accuracy на train: {train_accuracy:.2%}')
print(f'Accuracy на test:  {test_accuracy:.2%}')
print(f'Финальное значение MSE: {loss_values[-1]:.4f}')

## Графики результата

Показываем убывание MSE и границу, которую построил нейрон.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(loss_values, color='seagreen')
axes[0].set(title='MSE во время обучения', xlabel='Итерация', ylabel='MSE')
axes[0].grid(alpha=0.3)

x_min, x_max = X[:, 0].min() - 0.05, X[:, 0].max() + 0.05
y_min, y_max = X[:, 1].min() - 0.05, X[:, 1].max() + 0.05
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
grid_raw = np.c_[xx.ravel(), yy.ravel()]
grid = (grid_raw - mean) / std
grid_prediction = neuron.predict(grid).reshape(xx.shape)

axes[1].contourf(xx, yy, grid_prediction, alpha=0.25, cmap='coolwarm')
axes[1].scatter(X_test_raw[:, 0], X_test_raw[:, 1], c=y_test.ravel(), cmap='coolwarm', edgecolor='black')
axes[1].set(title=f'Тестовая выборка, accuracy = {test_accuracy:.2%}', xlabel='Желтизна', ylabel='Симметричность')
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

## Вывод

Реализован нейрон с функцией активации `tanh`, отличной от пороговой функции и сигмоиды из ЛР3. На фиксированном разбиении он получает **99,5% accuracy на тестовой выборке**. График MSE убывает, а граница классификации корректно разделяет два класса.